# Ventana Andina
## Predicción retrospectiva de ventanas frías en Huancané
Grupo 9 · 1INF24. Ejecutar todas las celdas. El modo rápido incorpora la celda diaria extraída de los NetCDF oficiales de 0,10°, sin inventar datos ni requerir permisos de Drive. El modo Drive permite repetir la extracción original. Los archivos de 0,05° no se usan. El entrenamiento y las decisiones se fijan antes de la prueba. Los resultados son térmicos, no una validación de lotes de chuño.

In [ ]:
%pip install -q numpy==2.5.4 pandas==3.1.1 scikit-learn==1.9.1 netCDF4==1.7.5.1.post1 xarray==2026.9.0 matplotlib==3.11.2 joblib==1.6.0 

In [ ]:
from pathlib import Path
import base64,gzip,json,hashlib,shutil
import pandas as pd
MODE="rapido" # cambiar a "drive" para leer los NetCDF de la carpeta del grupo
OUT=Path("resultados_reproducidos")
OUT.mkdir(exist_ok=True)

## Código completo del experimento
Las funciones son las mismas que se ejecutaron localmente. El CSV rápido conserva 17 cifras significativas y su procedencia y hashes están incluidos.

In [ ]:
Path("experimento.py").write_text('"""Experimento reproducible: ventanas frias en una celda PISCOt de Huancane.\n\nFases separadas: prepare selecciona TODO sin evaluar 2016--2020 y escribe\nfreeze.json; test exige ese archivo y modelos ya ajustados. No se reentrena\ncon validacion. --data-dir contiene los dos NetCDF diarios originales.\n"""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\nimport time\nimport warnings\nfrom pathlib import Path\n\nimport joblib\nimport netCDF4\nimport numpy as np\nimport pandas as pd\nimport sklearn\nfrom sklearn.exceptions import ConvergenceWarning\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import average_precision_score\nfrom sklearn.neighbors import KNeighborsClassifier\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.tree import DecisionTreeClassifier\n\nSEED = 2026\nTHERMALS = [-3., -2., -4.]\nMODELS = [\'logistica\', \'arbol\', \'knn\', \'mezcla\', \'frecuencia_mensual\', \'persistencia\']\nCLASSIFIERS = MODELS[:4]\nFEATURES = [f\'{v}_lag_{lag}\' for v in [\'tmin\', \'tmax\'] for lag in range(6, -1, -1)] + [\'sin_dia\', \'cos_dia\']\nFOLDS = [(1981, 1995, 1996, 2000), (1981, 2000, 2001, 2005), (1981, 2005, 2006, 2010)]\nGRIDS = {\n    \'logistica\': [{\'C\': c} for c in [.1, 1., 10.]],\n    \'arbol\': [{\'max_depth\': d, \'min_samples_leaf\': leaf} for d in [3, 5, 8] for leaf in [30, 10]],\n    \'knn\': [{\'n_neighbors\': k} for k in [31, 15, 5]],\n}\n\n\ndef clean_json(obj):\n    if isinstance(obj, dict):\n        return {str(k): clean_json(v) for k, v in obj.items()}\n    if isinstance(obj, (list, tuple)):\n        return [clean_json(v) for v in obj]\n    if isinstance(obj, np.ndarray):\n        return clean_json(obj.tolist())\n    if isinstance(obj, (np.integer,)):\n        return int(obj)\n    if isinstance(obj, (np.bool_,)):\n        return bool(obj)\n    if isinstance(obj, (np.floating, float)):\n        return float(obj) if np.isfinite(obj) else None\n    return obj\n\n\ndef write_json(path, obj):\n    Path(path).write_text(json.dumps(clean_json(obj), ensure_ascii=False, indent=2, sort_keys=True, allow_nan=False), encoding=\'utf-8\')\n\n\ndef sha(path):\n    with open(path, \'rb\') as f:\n        return hashlib.file_digest(f, \'sha256\').hexdigest()\n\n\ndef save_csv(df, path):\n    df.to_csv(path, index=False, encoding=\'utf-8-sig\', float_format=\'%.12g\', date_format=\'%Y-%m-%d\')\n\n\ndef extract_cell(data_dir, out):\n    """Lee solo una celda de cada NetCDF; no interpreta metricas del test."""\n    out.mkdir(parents=True, exist_ok=True)\n    frames, manifest, geometry = {}, {}, {}\n    for variable in [\'tmin\', \'tmax\']:\n        path = data_dir / f\'{variable}_daily_1981_2020_010.nc\'\n        manifest[variable] = {\'filename\': path.name, \'bytes\': path.stat().st_size, \'sha256\': sha(path)}\n        with netCDF4.Dataset(path) as ds:\n            lat, lon = np.asarray(ds[\'latitude\'][:]), np.asarray(ds[\'longitude\'][:])\n            i, j = int(np.argmin(abs(lat + 15.225))), int(np.argmin(abs(lon + 69.755)))\n            assert np.isclose(lat[i], -15.225) and np.isclose(lon[j], -69.755), \'Celda inesperada\'\n            dates = netCDF4.num2date(ds[\'time\'][:], ds[\'time\'].units, calendar=ds[\'time\'].calendar)\n            dates = pd.DatetimeIndex([d.strftime(\'%Y-%m-%d\') for d in dates])\n            vals = np.ma.filled(ds[variable][:, i, j], np.nan).astype(float)\n            frames[variable] = pd.Series(vals, index=dates)\n            geometry[variable] = {\'latitude\': float(lat[i]), \'longitude\': float(lon[j]), \'latitude_index\': i, \'longitude_index\': j, \'variable_dimensions\': list(ds[variable].dimensions), \'shape\': list(ds[variable].shape), \'calendar\': str(ds[\'time\'].calendar), \'global_attributes\': {a: str(ds.getncattr(a)) for a in ds.ncattrs()}}\n    a, b = frames[\'tmin\'], frames[\'tmax\']\n    assert a.index.equals(b.index), \'Fechas Tmin y Tmax distintas\'\n    duplicates = int(a.index.duplicated().sum())\n    audit = {\'duplicate_dates\': duplicates, \'dates_sorted\': bool(a.index.is_monotonic_increasing), \'first_date\': str(a.index.min().date()), \'last_date\': str(a.index.max().date()), \'rows_original\': len(a), \'geometry\': geometry}\n    write_json(out / \'manifest_fuentes.json\', manifest)\n    if duplicates or not a.index.is_monotonic_increasing:\n        write_json(out / \'auditoria_datos.json\', audit)\n        raise ValueError(\'Fechas duplicadas/desordenadas: no resolver silenciosamente\')\n    full = pd.date_range(\'1981-01-01\', \'2020-12-31\', freq=\'D\')\n    assert a.index.min() >= full.min() and a.index.max() <= full.max()\n    df = pd.DataFrame({\'tmin\': a, \'tmax\': b}).reindex(full)\n    df.index.name = \'fecha\'\n    finite = np.isfinite(df.tmin) & np.isfinite(df.tmax)\n    consistent = df.tmin <= df.tmax\n    df[\'valido_dia\'] = finite & consistent\n    audit.update({\'rows_full_calendar\': len(df), \'missing_calendar_dates\': len(full.difference(a.index)), \'missing_or_nonfinite_tmin\': int((~np.isfinite(df.tmin)).sum()), \'missing_or_nonfinite_tmax\': int((~np.isfinite(df.tmax)).sum()), \'tmin_exceeds_tmax\': int((finite & ~consistent).sum()), \'invalid_days\': int((~df.valido_dia).sum()), \'imputation\': \'Ninguna; las ventanas que tocan dias invalidos se excluyen.\', \'extraction_precision\': \'float64 a partir de valores originales NetCDF, sin redondear para entrenar.\'})\n    write_json(out / \'auditoria_datos.json\', audit)\n    # 17 cifras significativas garantizan round-trip de float64 en el cache.\n    df.reset_index().to_csv(out / \'celda_diaria.csv\', index=False, float_format=\'%.17g\', date_format=\'%Y-%m-%d\', encoding=\'utf-8\')\n    return df, manifest, audit\n\n\ndef make_examples(daily, thermal, min_year=1981, max_year=2020):\n    """t-6..t son atributos; t+1..t+3 son objetivo. QC en diez dias."""\n    x = pd.DataFrame(index=daily.index)\n    for v in [\'tmin\', \'tmax\']:\n        for lag in range(6, -1, -1):\n            x[f\'{v}_lag_{lag}\'] = daily[v].shift(lag)\n    denominator = np.where(daily.index.is_leap_year, 366., 365.)\n    angle = 2 * np.pi * daily.index.dayofyear.to_numpy() / denominator\n    x[\'sin_dia\'], x[\'cos_dia\'] = np.sin(angle), np.cos(angle)\n    history_valid = pd.concat([daily.valido_dia.shift(k) for k in range(7)], axis=1).eq(True).all(axis=1)\n    target_valid = pd.concat([daily.valido_dia.shift(-k) for k in range(1, 4)], axis=1).eq(True).all(axis=1)\n    future = pd.concat([daily.tmin.shift(-k) for k in range(1, 4)], axis=1)\n    past3 = pd.concat([daily.tmin.shift(k) for k in range(3)], axis=1)\n    x[\'valido_historia\'], x[\'valido_objetivo\'] = history_valid, target_valid\n    x[\'valido\'] = history_valid & target_valid\n    x[\'y\'] = future.le(thermal).all(axis=1).astype(float).where(x.valido)\n    x[\'persistencia\'] = past3.le(thermal).all(axis=1).astype(float).where(x.valido)\n    x[\'objetivo_inicio\'] = x.index + pd.Timedelta(days=1)\n    x[\'objetivo_fin\'] = x.index + pd.Timedelta(days=3)\n    x[\'year\'], x[\'month\'] = x.index.year, x.index.month\n    mmdd = x.index.strftime(\'%m-%d\')\n    x = x.loc[(mmdd >= \'04-01\') & (mmdd <= \'09-27\') & (x.year >= min_year) & (x.year <= max_year)].copy()\n    x.index.name = \'fecha\'\n    assert len(x) == 180 * (max_year - min_year + 1)\n    assert x.objetivo_fin.dt.month.le(9).all()\n    assert (x.objetivo_inicio > x.index).all()\n    assert np.isfinite(x.loc[x.valido, FEATURES].to_numpy()).all()\n    return x\n\n\ndef score(y, p, pred, beta=.5):\n    y, p, pred = np.asarray(y, dtype=int), np.asarray(p, dtype=float), np.asarray(pred, dtype=bool)\n    n = len(y)\n    tp = int(((y == 1) & pred).sum()); fp = int(((y == 0) & pred).sum())\n    fn = int(((y == 1) & ~pred).sum()); tn = int(((y == 0) & ~pred).sum())\n    positive = tp + fn\n    precision = tp / (tp + fp) if tp + fp else 0.\n    recall = tp / positive if positive else np.nan\n    def f(b):\n        return (1+b*b)*tp / ((1+b*b)*tp + fp + b*b*fn) if positive else np.nan\n    return {\'n\': n, \'positivos\': positive, \'prevalencia\': positive/n if n else np.nan, \'avisos\': tp+fp, \'VP\': tp, \'FP\': fp, \'FN\': fn, \'VN\': tn, \'precision\': precision, \'recall\': recall, \'F0.5\': f(.5), \'F1\': f(1.), \'Fbeta\': f(beta), \'Brier\': float(np.mean((p-y)**2)) if n else np.nan, \'average_precision\': float(average_precision_score(y, p)) if positive else np.nan}\n\n\ndef choose_threshold(y, probs, beta):\n    """Empates numericos <=1e-12 favorecen tau mayor; aviso p>=tau."""\n    candidates = np.r_[np.unique(probs), np.nextafter(float(np.max(probs)), np.inf)]\n    rows = []\n    for tau in candidates:\n        s = score(y, probs, probs >= tau, beta)\n        rows.append({\'tau\': float(tau), \'Fbeta\': s[\'Fbeta\'], \'precision\': s[\'precision\'], \'recall\': s[\'recall\'], \'VP\': s[\'VP\'], \'FP\': s[\'FP\'], \'FN\': s[\'FN\']})\n    table = pd.DataFrame(rows)\n    if not np.isfinite(table.Fbeta).any():\n        raise ValueError(\'No hay positivos en validacion; no puede ajustarse Fbeta\')\n    best = table.Fbeta.max()\n    selected = table.loc[np.isclose(table.Fbeta, best, atol=1e-12, rtol=0)].tau.max()\n    return float(selected), table\n\n\ndef model_factory(name, cfg):\n    if name == \'logistica\':\n        # L2 es el valor por defecto con lbfgs; explicitado si la API lo permite.\n        return make_pipeline(StandardScaler(), LogisticRegression(C=cfg[\'C\'], solver=\'lbfgs\', max_iter=10000, random_state=SEED))\n    if name == \'arbol\':\n        return DecisionTreeClassifier(criterion=\'gini\', random_state=SEED, **cfg)\n    return make_pipeline(StandardScaler(), KNeighborsClassifier(weights=\'uniform\', metric=\'euclidean\', algorithm=\'brute\', n_jobs=1, **cfg))\n\n\ndef select_config(name, frame, thermal):\n    rows, candidate_rows = [], []\n    for cfg_idx, cfg in enumerate(GRIDS[name]):\n        bs = []\n        for fold, (train_start, train_end, eval_start, eval_end) in enumerate(FOLDS, 1):\n            tr = frame.loc[frame.valido & frame.year.between(train_start, train_end)]\n            va = frame.loc[frame.valido & frame.year.between(eval_start, eval_end)]\n            assert tr.objetivo_fin.max() < va.index.min()\n            model = model_factory(name, cfg)\n            with warnings.catch_warnings():\n                warnings.simplefilter(\'error\', ConvergenceWarning)\n                model.fit(tr[FEATURES], tr.y.astype(int))\n            if name != \'arbol\':\n                assert np.allclose(model[0].mean_, tr[FEATURES].mean().to_numpy(), atol=1e-10)\n            p = model.predict_proba(va[FEATURES])[:, 1]\n            brier = float(np.mean((p - va.y.to_numpy())**2))\n            bs.append(brier)\n            rows.append({\'thermal\': thermal, \'model\': name, \'candidate\': cfg_idx, \'params\': json.dumps(cfg, sort_keys=True), \'fold\': fold, \'train_start\': train_start, \'train_end\': train_end, \'eval_start\': eval_start, \'eval_end\': eval_end, \'n_train\': len(tr), \'n_eval\': len(va), \'Brier\': brier})\n        candidate_rows.append({\'thermal\': thermal, \'model\': name, \'candidate\': cfg_idx, \'params\': json.dumps(cfg, sort_keys=True), \'Brier_mean\': float(np.mean(bs)), \'Brier_std_population\': float(np.std(bs))})\n    best_score = min(r[\'Brier_mean\'] for r in candidate_rows)\n    tied = [r for r in candidate_rows if np.isclose(r[\'Brier_mean\'], best_score, atol=1e-12, rtol=0)]\n    # Orden de la rejilla: menor C, menor profundidad/mayor hoja, mayor k.\n    winner = min(tied, key=lambda r: r[\'candidate\'])\n    for r in candidate_rows:\n        r[\'selected\'] = r[\'candidate\'] == winner[\'candidate\']\n    return GRIDS[name][winner[\'candidate\']], rows, candidate_rows\n\n\ndef select_principal(metrics):\n    best_f = max(metrics[m][\'Fbeta\'] for m in CLASSIFIERS)\n    tied = [m for m in CLASSIFIERS if np.isclose(metrics[m][\'Fbeta\'], best_f, atol=1e-12, rtol=0)]\n    best_b = min(metrics[m][\'Brier\'] for m in tied)\n    return next(m for m in tied if np.isclose(metrics[m][\'Brier\'], best_b, atol=1e-12, rtol=0))\n\n\ndef episodes(daily, thermal, first_year, last_year):\n    """Rachas maximales dentro de abril-septiembre; episodios son >=3 dias.\n\n    Un episodio de L dias contiene L-2 subventanas completas, pero las\n    emisiones cubren objetivos desde 2/abril: documentar esta frontera.\n    """\n    rows = []\n    for year in range(first_year, last_year+1):\n        d = daily.loc[f\'{year}-04-01\':f\'{year}-09-30\']\n        cold = (d.valido_dia & (d.tmin <= thermal)).to_numpy()\n        starts = np.flatnonzero(cold & ~np.r_[False, cold[:-1]])\n        ends = np.flatnonzero(cold & ~np.r_[cold[1:], False])\n        for start, end in zip(starts, ends):\n            length = int(end-start+1)\n            if length >= 3:\n                rows.append({\'thermal\': thermal, \'year\': year, \'inicio\': str(d.index[start].date()), \'fin\': str(d.index[end].date()), \'dias\': length, \'tmin_min\': float(d.tmin.iloc[start:end+1].min()), \'subventanas_tres_dias\': length-2, \'toca_borde_temporada\': bool(start==0 or end==len(d)-1)})\n    return pd.DataFrame(rows, columns=[\'thermal\',\'year\',\'inicio\',\'fin\',\'dias\',\'tmin_min\',\'subventanas_tres_dias\',\'toca_borde_temporada\'])\n\n\ndef sanity_tests(daily):\n    # Modificar el futuro no puede cambiar atributos del presente.\n    ref = make_examples(daily.loc[:\'2015-12-31\'], -3, 1981, 2015)\n    d = pd.Timestamp(\'2010-06-15\')\n    changed = daily.loc[:\'2015-12-31\'].copy()\n    changed.loc[d+pd.Timedelta(days=1):, [\'tmin\',\'tmax\']] += 100\n    new = make_examples(changed, -3, 1981, 2015)\n    assert np.array_equal(ref.loc[d, FEATURES].to_numpy(), new.loc[d, FEATURES].to_numpy())\n    assert ref.loc[d,\'objetivo_fin\'] == d+pd.Timedelta(days=3)\n    assert list(ref.loc[d, [f\'tmin_lag_{i}\' for i in range(6,-1,-1)]]) == list(daily.loc[d-pd.Timedelta(days=6):d, \'tmin\'])\n    assert all(make_examples(daily, -3, y,y).shape[0] == 180 for y in [1981,1984,2016,2020])\n    s = score([1,1,0], [.8,.2,.7], [1,0,1])\n    assert np.isclose(s[\'F0.5\'], 1.25/(1.25+1+.25))\n    z = score([0,0], [.1,.2], [0,0])\n    assert z[\'precision\']==0 and np.isnan(z[\'recall\']) and np.isnan(z[\'F0.5\'])\n    # QC: un dia invalido invalida t-6..t+3, sin rellenar el objetivo.\n    bad = daily.copy(); bad.loc[d,\'valido_dia\'] = False\n    check = make_examples(bad, -3, 2010, 2010)\n    assert not check.loc[d-pd.Timedelta(days=3):d+pd.Timedelta(days=6),\'valido\'].any()\n    assert check.loc[d-pd.Timedelta(days=4),\'valido\'] == make_examples(daily,-3,2010,2010).loc[d-pd.Timedelta(days=4),\'valido\']\n    return {\'future_perturbation_does_not_change_features\': True, \'exact_feature_alignment\': True, \'three_future_days\': True, \'180_candidate_dates_per_year\': True, \'Fbeta_formula\': True, \'no_positives_convention\': True, \'invalid_full_window_no_imputation\': True, \'temporal_folds_no_overlap_and_train_scalers_checked_during_fit\': True}\n\n\ndef prepare(data_dir, out):\n    begin = time.perf_counter()\n    out.mkdir(parents=True, exist_ok=True)\n    daily, manifest, audit = extract_cell(data_dir, out)\n    tests = sanity_tests(daily)\n    write_json(out/\'sanity_tests.json\', tests)\n    # No ejemplos, etiquetas ni probabilidades de prueba durante seleccion.\n    pretest_daily = daily.loc[:\'2015-12-31\'].copy()\n    all_cv, all_means, all_val, all_alpha, all_thresholds, all_pop, all_month, all_eps = [], [], [], [], [], [], [], []\n    state, decisions = {}, {}\n    for thermal in THERMALS:\n        print(f\'PREPARE thermal={thermal:g}; solo 1981-2015\', flush=True)\n        frame = make_examples(pretest_daily, thermal, 1981, 2015)\n        train = frame.loc[frame.valido & frame.year.le(2010)]\n        val = frame.loc[frame.valido & frame.year.ge(2011)]\n        configs, fitted = {}, {}\n        for name in GRIDS:\n            cfg, cvrows, means = select_config(name, frame, thermal)\n            configs[name] = cfg; all_cv.extend(cvrows); all_means.extend(means)\n            fitted[name] = model_factory(name, cfg)\n            with warnings.catch_warnings():\n                warnings.simplefilter(\'error\', ConvergenceWarning)\n                fitted[name].fit(train[FEATURES], train.y.astype(int))\n        probs = {n: fitted[n].predict_proba(val[FEATURES])[:,1] for n in fitted}\n        monthly = train.groupby(\'month\').y.mean().to_dict()\n        probs[\'frecuencia_mensual\'] = val.month.map(monthly).to_numpy()\n        probs[\'persistencia\'] = val.persistencia.to_numpy()\n        arows = []\n        for alpha in [0., .25, .5, .75, 1.]:\n            p = alpha*probs[\'logistica\'] + (1-alpha)*probs[\'arbol\']\n            arows.append({\'thermal\': thermal, \'alpha\': alpha, \'Brier\': float(np.mean((p-val.y.to_numpy())**2))})\n        low = min(r[\'Brier\'] for r in arows)\n        alpha = max(r[\'alpha\'] for r in arows if np.isclose(r[\'Brier\'],low,atol=1e-12,rtol=0))\n        for r in arows: r[\'selected\'] = r[\'alpha\']==alpha\n        all_alpha.extend(arows)\n        probs[\'mezcla\'] = alpha*probs[\'logistica\'] + (1-alpha)*probs[\'arbol\']\n        if alpha in [0.,1.]:\n            assert np.array_equal(probs[\'mezcla\'], probs[\'logistica\' if alpha==1 else \'arbol\'])\n        decisions_by_beta = {}\n        for beta in [.5, 1.]:\n            thresholds, metrics = {}, {}\n            for name in MODELS:\n                if name == \'persistencia\':\n                    tau = .5\n                else:\n                    tau, table = choose_threshold(val.y.to_numpy(), probs[name], beta)\n                    table[\'thermal\'],table[\'beta\'],table[\'model\']=thermal,beta,name\n                    all_thresholds.extend(table.to_dict(\'records\'))\n                thresholds[name] = tau\n                metrics[name] = score(val.y, probs[name], probs[name]>=tau, beta)\n                all_val.append({\'thermal\':thermal,\'beta\':beta,\'model\':name,\'tau\':tau,**metrics[name]})\n            principal = select_principal(metrics)\n            decisions_by_beta[str(beta)] = {\'thresholds\':thresholds,\'principal\':principal,\'validation_metrics\':metrics}\n        key = str(int(thermal))\n        decisions[key] = {\'thermal\':thermal,\'selected_configs\':configs,\'alpha\':alpha,\'monthly_training_frequencies\':monthly,\'decisions_by_beta\':decisions_by_beta,\'training_n\':len(train),\'validation_n\':len(val)}\n        state[key] = {\'models\':fitted,\'monthly\':monthly,\'validation_dates\':val.index,\'validation_probs\':probs}\n        pred_val = val[[\'y\',\'persistencia\',\'objetivo_inicio\',\'objetivo_fin\']].copy()\n        for n in MODELS: pred_val[f\'p_{n}\'] = probs[n]\n        save_csv(pred_val.reset_index(),out/f\'predicciones_validacion_m{abs(int(thermal))}.csv\')\n        for label,a,b in [(\'entrenamiento\',1981,2010),(\'validacion\',2011,2015)]:\n            sub=frame.loc[frame.year.between(a,b)]\n            all_pop.append({\'thermal\':thermal,\'split\':label,\'candidate_n\':len(sub),\'valid_n\':int(sub.valido.sum()),\'invalid_n\':int((~sub.valido).sum()),\'positive_n\':int(sub.y.sum()),\'prevalence\':float(sub.y.mean())})\n        for month,g in train.groupby(\'month\'):\n            all_month.append({\'thermal\':thermal,\'month\':month,\'n\':len(g),\'positive_n\':int(g.y.sum()),\'window_frequency\':float(g.y.mean()),\'tmin_mean\':float(g.tmin_lag_0.mean()),\'tmin_min\':float(g.tmin_lag_0.min()),\'tmax_mean\':float(g.tmax_lag_0.mean())})\n        all_eps.append(episodes(pretest_daily,thermal,1981,2015))\n        print(f"  configs={configs}; alpha={alpha}; principal={decisions_by_beta[\'0.5\'][\'principal\']}",flush=True)\n    write_json(out/\'freeze.json\',{\'protocol_version\':\'1.0\',\'seed\':SEED,\'feature_order\':FEATURES,\'thermal_definitions\':THERMALS,\'train\':\'1981-2010\',\'validation\':\'2011-2015\',\'test\':\'2016-2020 (NO EVALUADO EN ESTA FASE)\',\'model_fit_uses_validation\':False,\'season_emission\':\'04-01..09-27 inclusive; 180/y\',\'target\':\'all three tmin(t+1:t+3) <= thermal\',\'seasonal_encoding\':\'sin/cos(2*pi*dayofyear/D), D=365 or 366 of emission year\',\'alpha_formula\':\'alpha*LR+(1-alpha)*tree\',\'alpha_tie\':\'higher alpha\',\'tie_tolerance_absolute\':1e-12,\'threshold_rule\':\'p>=tau; candidates unique validation probabilities + nextafter(max,+inf); Fbeta tie higher tau\',\'config_tie\':\'lower C; lower depth then larger leaf; larger k\',\'principal_tie\':\'higher validation Fbeta, then lower Brier, then LR/tree/KNN/mix\',\'beta1_scope\':\'secondary; same configurations, fitted models, probabilities and alpha as beta0.5\',\'bootstrap\':\'1000 paired calendar moving-block samples; six blocks of30 per season within each of five fixed test years; shared mask and indices; seed2026\',\'sources\':manifest,\'code_sha256\':sha(__file__),\'versions\':{\'python\':platform.python_version(),\'numpy\':np.__version__,\'pandas\':pd.__version__,\'sklearn\':sklearn.__version__},\'decisions\':decisions})\n    joblib.dump(state,out/\'modelos_congelados.joblib\',compress=3)\n    for name,rows in [(\'cv_folds\',all_cv),(\'cv_resumen\',all_means),(\'metricas_validacion\',all_val),(\'alpha_validacion\',all_alpha),(\'busqueda_umbrales_validacion\',all_thresholds),(\'prevalencia_pretest\',all_pop),(\'eda_mensual_entrenamiento\',all_month)]:\n        save_csv(pd.DataFrame(rows),out/f\'{name}.csv\')\n    save_csv(pd.concat(all_eps,ignore_index=True),out/\'episodios_pretest.csv\')\n    write_json(out/\'timing_prepare.json\',{\'seconds\':time.perf_counter()-begin})\n    print(\'FREEZE ESCRITO. La fase prepare termina sin evaluar la prueba.\',flush=True)\n\n\ndef bootstrap_test(frame, probs, decision, thermal, beta, n_boot=1000):\n    """Remuestreo por calendario antes de la mascara, pareado entre metodos."""\n    rng = np.random.default_rng(SEED)\n    years = sorted(frame.year.unique())\n    assert len(years)==5 and len(frame)==900\n    y = frame.y.to_numpy(); valid=frame.valido.to_numpy()\n    allp = {n:np.asarray(probs[n],float) for n in MODELS}\n    thresholds = decision[\'thresholds\']; principal = decision[\'principal\']\n    rows, deltas = [], []\n    draws = np.empty((n_boot,5,6),dtype=np.int16)\n    # Inicios0..150 inclusive: ningun bloque cruza septiembre o cambia de ano.\n    for rep in range(n_boot):\n        picks=[]\n        for j,yr in enumerate(years):\n            locations = np.flatnonzero(frame.year.to_numpy()==yr)\n            assert len(locations)==180\n            starts=rng.integers(0,151,size=6);draws[rep,j]=starts\n            picks.extend(locations[(starts[:,None]+np.arange(30)).ravel()].tolist())\n        idx=np.asarray(picks,dtype=int);idx=idx[valid[idx]]\n        scores={}\n        for name in MODELS:\n            p=allp[name][idx]\n            scores[name]=score(y[idx],p,p>=thresholds[name],beta)\n            rows.append({\'replica\':rep,\'thermal\':thermal,\'beta\':beta,\'model\':name,**{k:scores[name][k] for k in [\'n\',\'positivos\',\'precision\',\'recall\',\'F0.5\',\'F1\',\'Fbeta\',\'Brier\']}})\n        for comparator in MODELS:\n            if comparator == principal:continue\n            deltas.append({\'replica\':rep,\'thermal\':thermal,\'beta\':beta,\'principal\':principal,\'comparador\':comparator,\'delta_F0.5\':scores[principal][\'F0.5\']-scores[comparator][\'F0.5\'],\'delta_F1\':scores[principal][\'F1\']-scores[comparator][\'F1\'],\'delta_Brier\':scores[principal][\'Brier\']-scores[comparator][\'Brier\']})\n        for comparator in [\'logistica\',\'arbol\']:\n            deltas.append({\'replica\':rep,\'thermal\':thermal,\'beta\':beta,\'principal\':\'mezcla\',\'comparador\':comparator,\'delta_F0.5\':scores[\'mezcla\'][\'F0.5\']-scores[comparator][\'F0.5\'],\'delta_F1\':scores[\'mezcla\'][\'F1\']-scores[comparator][\'F1\'],\'delta_Brier\':scores[\'mezcla\'][\'Brier\']-scores[comparator][\'Brier\']})\n    return pd.DataFrame(rows),pd.DataFrame(deltas).drop_duplicates([\'replica\',\'thermal\',\'beta\',\'principal\',\'comparador\']),draws\n\n\ndef intervals(table, groupcols, values):\n    rows=[]\n    for key,g in table.groupby(groupcols,dropna=False):\n        key=key if isinstance(key,tuple) else (key,)\n        for value in values:\n            arr=g[value].to_numpy();valid=arr[np.isfinite(arr)]\n            rows.append({**dict(zip(groupcols,key)),\'metric\':value,\'lower_2.5\':float(np.percentile(valid,2.5)) if len(valid) else np.nan,\'upper_97.5\':float(np.percentile(valid,97.5)) if len(valid) else np.nan,\'replicas_estimables\':len(valid),\'replicas_totales\':len(g),\'fraction_not_estimable\':1-len(valid)/len(g)})\n    return pd.DataFrame(rows)\n\n\ndef test(out):\n    begin=time.perf_counter()\n    frozen=json.loads((out/\'freeze.json\').read_text(encoding=\'utf-8\'))\n    assert frozen[\'code_sha256\']==sha(__file__),\'El codigo cambio despues de congelar decisiones\'\n    state=joblib.load(out/\'modelos_congelados.joblib\')\n    daily=pd.read_csv(out/\'celda_diaria.csv\',parse_dates=[\'fecha\'],float_precision=\'round_trip\').set_index(\'fecha\')\n    allmetrics,allannual,allpop,eps=[],[],[],[]\n    boot_tables,delta_tables=[],[]\n    for thermal in THERMALS:\n        print(f\'TEST thermal={thermal:g}; ninguna decision se reajusta\',flush=True)\n        key=str(int(thermal));f=frozen[\'decisions\'][key];st=state[key]\n        frame=make_examples(daily,thermal,2016,2020)\n        valid=frame.valido.to_numpy();sub=frame.loc[frame.valido]\n        probs={}\n        for n,m in st[\'models\'].items():\n            probs[n]=np.full(len(frame),np.nan)\n            probs[n][valid]=m.predict_proba(sub[FEATURES])[:,1]\n        probs[\'frecuencia_mensual\']=frame.month.map(st[\'monthly\']).to_numpy(dtype=float)\n        probs[\'persistencia\']=frame.persistencia.to_numpy()\n        probs[\'mezcla\']=f[\'alpha\']*probs[\'logistica\']+(1-f[\'alpha\'])*probs[\'arbol\']\n        if f[\'alpha\'] in [0.,1.]:\n            assert np.array_equal(probs[\'mezcla\'][valid],probs[\'logistica\' if f[\'alpha\']==1 else \'arbol\'][valid])\n        predictions=frame[[\'objetivo_inicio\',\'objetivo_fin\',\'year\',\'month\',\'valido_historia\',\'valido_objetivo\',\'valido\',\'y\']].copy()\n        for name in MODELS:predictions[f\'p_{name}\']=probs[name]\n        for beta in [.5,1.]:\n            dec=f[\'decisions_by_beta\'][str(beta)]\n            for name in MODELS:\n                tau=dec[\'thresholds\'][name]\n                prediction=probs[name]>=tau\n                predictions[f\'aviso_beta{beta}_{name}\']=pd.Series(prediction.astype(float),index=frame.index).where(frame.valido)\n                allmetrics.append({\'thermal\':thermal,\'beta\':beta,\'model\':name,\'principal\':name==dec[\'principal\'],\'tau\':tau,**score(frame.y[valid],probs[name][valid],prediction[valid],beta)})\n                for year in range(2016,2021):\n                    mask=valid&(frame.year.to_numpy()==year)\n                    allannual.append({\'thermal\':thermal,\'beta\':beta,\'model\':name,\'year\':year,\'principal\':name==dec[\'principal\'],**score(frame.y[mask],probs[name][mask],prediction[mask],beta)})\n            b,d,draws=bootstrap_test(frame,probs,dec,thermal,beta)\n            boot_tables.append(b);delta_tables.append(d)\n            if thermal==-3 and beta==.5:np.save(out/\'bootstrap_inicios_calendario.npy\',draws)\n        save_csv(predictions.reset_index(),out/f\'predicciones_prueba_m{abs(int(thermal))}.csv\')\n        allpop.append({\'thermal\':thermal,\'split\':\'prueba\',\'candidate_n\':len(frame),\'valid_n\':int(valid.sum()),\'invalid_n\':int((~valid).sum()),\'positive_n\':int(frame.y.sum()),\'prevalence\':float(frame.y.mean())})\n        eps.append(episodes(daily,thermal,2016,2020))\n    metrics=pd.DataFrame(allmetrics);annual=pd.DataFrame(allannual)\n    boot=pd.concat(boot_tables,ignore_index=True);deltas=pd.concat(delta_tables,ignore_index=True)\n    save_csv(metrics,out/\'metricas_prueba.csv\');save_csv(annual,out/\'metricas_prueba_anuales.csv\')\n    save_csv(pd.DataFrame(allpop),out/\'prevalencia_prueba.csv\')\n    save_csv(pd.concat(eps,ignore_index=True),out/\'episodios_prueba.csv\')\n    save_csv(boot,out/\'bootstrap_metricas_1000.csv\');save_csv(deltas,out/\'bootstrap_diferencias_1000.csv\')\n    save_csv(intervals(boot,[\'thermal\',\'beta\',\'model\'],[\'precision\',\'recall\',\'F0.5\',\'F1\',\'Brier\']),out/\'bootstrap_intervalos_metricas.csv\')\n    save_csv(intervals(deltas,[\'thermal\',\'beta\',\'principal\',\'comparador\'],[\'delta_F0.5\',\'delta_F1\',\'delta_Brier\']),out/\'bootstrap_intervalos_diferencias.csv\')\n    # Hipotesis fijadas: comparar puntos estrictamente, sin llamarlos significativos.\n    conclusions={}\n    for thermal in THERMALS:\n        key=str(int(thermal));sel=frozen[\'decisions\'][key][\'decisions_by_beta\'][\'0.5\'][\'principal\']\n        g=metrics.loc[(metrics.thermal==thermal)&(metrics.beta==.5)].set_index(\'model\')\n        diffs={ref:float(g.loc[sel,\'F0.5\']-g.loc[ref,\'F0.5\']) for ref in [\'persistencia\',\'frecuencia_mensual\']}\n        md={ref:float(g.loc[\'mezcla\',\'Brier\']-g.loc[ref,\'Brier\']) for ref in [\'logistica\',\'arbol\']}\n        conclusions[key]={\'principal\':sel,\'alpha\':frozen[\'decisions\'][key][\'alpha\'],\'delta_F0.5_vs_referencias\':diffs,\'hipotesis_principal_apoyada_puntualmente\':all(v>0 for v in diffs.values()),\'delta_Brier_mezcla_vs_componentes\':md,\'hipotesis_complementaria_apoyada_puntualmente\':all(v<0 for v in md.values()),\'scope\':\'Puntos historicos; intervalos bootstrap exploratorios, no prueba de significancia ni validacion productiva.\'}\n    write_json(out/\'conclusiones_numericas.json\',conclusions)\n    write_json(out/\'timing_test.json\',{\'seconds\':time.perf_counter()-begin,\'freeze_sha256\':sha(out/\'freeze.json\'),\'bootstrap_replicates\':1000})\n    print(metrics.loc[(metrics.thermal==-3)&(metrics.beta==.5)].to_string(index=False),flush=True)\n    print(json.dumps(clean_json(conclusions[\'-3\']),ensure_ascii=False,indent=2),flush=True)\n\n\ndef main():\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'phase\',choices=[\'prepare\',\'test\',\'all\'])\n    parser.add_argument(\'--data-dir\',type=Path,default=Path(\'data\'))\n    parser.add_argument(\'--out-dir\',type=Path,default=Path(\'resultados\'))\n    args=parser.parse_args()\n    if args.phase in [\'prepare\',\'all\']:prepare(args.data_dir,args.out_dir)\n    if args.phase in [\'test\',\'all\']:test(args.out_dir)\n\n\nif __name__==\'__main__\':main()\n',encoding="utf-8")
import experimento as exp

In [ ]:
CSV_BYTES=gzip.decompress(base64.b64decode('...'))
assert hashlib.sha256(CSV_BYTES).hexdigest()=='51cb3e6d931c2c8d15e2897aa6804ae2eb6657c7293fc5eb9cb896e23f8b8eae'
Path("celda_diaria_original.csv").write_bytes(CSV_BYTES)
MANIFEST={'tmax': {'bytes': 631950361, 'filename': 'tmax_daily_1981_2020_010.nc', 'sha256': 'd2f0a8dda879c8c49a892cea11cf51c206f1402b0b5ddf0aeb178e53e2f25f62'}, 'tmin': {'bytes': 649625447, 'filename': 'tmin_daily_1981_2020_010.nc', 'sha256': '2f655bbba2a45fa09fb66a0de04a3aed5ea21fb771bae1531076cae722805d66'}}
AUDIT={'dates_sorted': True, 'duplicate_dates': 0, 'extraction_precision': 'float64 a partir de valores originales NetCDF, sin redondear para entrenar.', 'first_date': '1981-01-01', 'geometry': {'tmax': {'calendar': 'proleptic_gregorian', 'global_attributes': {'Issues': 'https://github.com/adrHuerta/PISCOt_v1-2/issues', 'Variable': 'maximum temperature (Â°C)'}, 'latitude': -15.225, 'latitude_index': 164, 'longitude': -69.75500000000001, 'longitude_index': 116, 'shape': [14610, 199, 143], 'variable_dimensions': ['time', 'latitude', 'longitude']}, 'tmin': {'calendar': 'proleptic_gregorian', 'global_attributes': {'Issues': 'https://github.com/adrHuerta/PISCOt_v1-2/issues', 'Variable': 'minimum temperature (Â°C)'}, 'latitude': -15.225, 'latitude_index': 164, 'longitude': -69.75500000000001, 'longitude_index': 116, 'shape': [14610, 199, 143], 'variable_dimensions': ['time', 'latitude', 'longitude']}}, 'imputation': 'Ninguna; las ventanas que tocan dias invalidos se excluyen.', 'invalid_days': 0, 'last_date': '2020-12-31', 'missing_calendar_dates': 0, 'missing_or_nonfinite_tmax': 0, 'missing_or_nonfinite_tmin': 0, 'rows_full_calendar': 14610, 'rows_original': 14610, 'tmin_exceeds_tmax': 0}

In [ ]:
if MODE == "rapido":
    def extraer_cache(data_dir,out):
        out.mkdir(parents=True,exist_ok=True)
        df=pd.read_csv("celda_diaria_original.csv",parse_dates=["fecha"],float_precision="round_trip").set_index("fecha")
        assert len(df)==14610 and not df.index.duplicated().any()
        assert df.valido_dia.all() and (df.tmin<=df.tmax).all()
        (out/"celda_diaria.csv").write_bytes(CSV_BYTES)
        exp.write_json(out/"manifest_fuentes.json",MANIFEST)
        exp.write_json(out/"auditoria_datos.json",AUDIT)
        return df,MANIFEST,AUDIT
    exp.extract_cell=extraer_cache
    DATA=Path(".")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    matches=list(Path("/content/drive/MyDrive").rglob("tmin_daily_1981_2020_010.nc"))
    if not matches:
        raise FileNotFoundError("Agrega un acceso directo a la carpeta compartida en Mi unidad o indica DATA manualmente. Se necesitan Tmin y Tmax 010.")
    DATA=matches[0].parent
    assert (DATA/"tmax_daily_1981_2020_010.nc").exists()

## Ajuste y congelación
Esta fase usa únicamente 1981–2015 para decisiones. El archivo freeze.json registra modelos, parámetros y umbrales. No cambies el protocolo según los resultados posteriores.

In [ ]:
exp.prepare(DATA,OUT)
frozen=json.loads((OUT/"freeze.json").read_text())
print(json.dumps(frozen["decisions"]["-3"],indent=2))

## Prueba reservada y exportación
Se mantienen las decisiones de validación. Los intervalos son exploratorios. F1 es secundario y no reemplaza la hipótesis principal.

In [ ]:
exp.test(OUT)
Path("resumen_adicional.py").write_text('"""Descriptivos de salidas congeladas; no ajusta modelos, pesos ni umbrales."""\nfrom pathlib import Path\nimport json\nimport pandas as pd\n\n\ndef resumir(out):\n    out = Path(out)\n    avisos = pd.read_csv(out / \'predicciones_prueba_m3.csv\')\n    episodios = pd.read_csv(out / \'episodios_prueba.csv\')\n    episodios = episodios.loc[episodios.thermal == -3]\n    validos = avisos[\'valido\'].astype(bool)\n    a = avisos.loc[validos, \'aviso_beta0.5_logistica\']\n    b = avisos.loc[validos, \'aviso_beta0.5_arbol\']\n    resultado = {\n        \'alcance\': \'Descripcion posterior de salidas, sin nuevas decisiones de ajuste\',\n        \'thermal\': -3,\n        \'beta\': 0.5,\n        \'n\': int(validos.sum()),\n        \'desacuerdo_avisos_logistica_arbol\': int((a != b).sum()),\n        \'fraccion_desacuerdo\': float((a != b).mean()),\n        \'episodios_prueba\': len(episodios),\n        \'episodios_por_ano\': {str(int(k)): int(v) for k, v in episodios.groupby(\'year\').size().items()},\n        \'subventanas_en_episodios\': int(episodios.subventanas_tres_dias.sum()),\n        \'positivos_prueba\': int(avisos.loc[validos, \'y\'].sum()),\n        \'episodios_tocan_borde\': int(episodios.toca_borde_temporada.sum()),\n    }\n    (out / \'resumen_descriptivo.json\').write_text(\n        json.dumps(resultado, indent=2, ensure_ascii=False), encoding=\'utf-8\'\n    )\n    return resultado\n\n\nif __name__ == \'__main__\':\n    import argparse\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--out-dir\', type=Path, default=Path(\'resultados\'))\n    print(json.dumps(resumir(p.parse_args().out_dir), indent=2))\n',encoding="utf-8")
from resumen_adicional import resumir
print(json.dumps(resumir(OUT),indent=2))

# Comprobamos las cifras publicadas, despues de evaluar y sin cambiar decisiones.
from io import StringIO
referencia=pd.read_csv(StringIO('\ufeffthermal,beta,model,principal,tau,n,positivos,prevalencia,avisos,VP,FP,FN,VN,precision,recall,F0.5,F1,Fbeta,Brier,average_precision\n-3,0.5,logistica,True,0.551970003564,900,330,0.366666666667,347,272,75,58,495,0.78386167147,0.824242424242,0.791618160652,0.803545051699,0.791618160652,0.112431201147,0.82424191981\n-3,0.5,arbol,False,0.739495798319,900,330,0.366666666667,303,241,62,89,508,0.795379537954,0.730303030303,0.781452658885,0.761453396524,0.781452658885,0.122625183315,0.764164166709\n-3,0.5,knn,False,0.612903225806,900,330,0.366666666667,309,239,70,91,500,0.773462783172,0.724242424242,0.763090676884,0.748043818466,0.763090676884,0.123214244421,0.778877682946\n-3,0.5,mezcla,False,0.52676251896,900,330,0.366666666667,350,271,79,59,491,0.774285714286,0.821212121212,0.78323699422,0.797058823529,0.78323699422,0.113003361877,0.823842151769\n-3,0.5,frecuencia_mensual,False,0.714444444444,900,330,0.366666666667,305,207,98,123,472,0.67868852459,0.627272727273,0.667741935484,0.651968503937,0.667741935484,0.165874192073,0.661810835335\n-3,0.5,persistencia,False,0.5,900,330,0.366666666667,330,236,94,94,476,0.715151515152,0.715151515152,0.715151515152,0.715151515152,0.715151515152,0.208888888889,0.615886134068\n-3,1,logistica,False,0.436734568287,900,330,0.366666666667,404,295,109,35,461,0.730198019802,0.893939393939,0.757965056526,0.803814713896,0.803814713896,0.112431201147,0.82424191981\n-3,1,arbol,False,0.4375,900,330,0.366666666667,423,295,128,35,442,0.697399527187,0.893939393939,0.729475766568,0.783532536521,0.783532536521,0.122625183315,0.764164166709\n-3,1,knn,False,0.41935483871,900,330,0.366666666667,421,292,129,38,441,0.693586698337,0.884848484848,0.724925521351,0.777629826897,0.777629826897,0.123214244421,0.778877682946\n-3,1,mezcla,True,0.388272237192,900,330,0.366666666667,422,302,120,28,450,0.715639810427,0.915151515152,0.748265609514,0.803191489362,0.803191489362,0.113003361877,0.823842151769\n-3,1,frecuencia_mensual,False,0.454838709677,900,330,0.366666666667,460,266,194,64,376,0.578260869565,0.806060606061,0.612903225806,0.673417721519,0.673417721519,0.165874192073,0.661810835335\n-3,1,persistencia,False,0.5,900,330,0.366666666667,330,236,94,94,476,0.715151515152,0.715151515152,0.715151515152,0.715151515152,0.715151515152,0.208888888889,0.615886134068\n-2,0.5,logistica,False,0.7062403063,900,422,0.468888888889,360,311,49,111,429,0.863888888889,0.736966824645,0.835123523093,0.795396419437,0.835123523093,0.120035786506,0.879329920995\n-2,0.5,arbol,False,0.765449438202,900,422,0.468888888889,384,322,62,100,416,0.838541666667,0.763033175355,0.82226762002,0.799007444169,0.82226762002,0.128043427798,0.830093963746\n-2,0.5,knn,False,0.612903225806,900,422,0.468888888889,405,323,82,99,396,0.797530864198,0.765402843602,0.790891283056,0.781136638452,0.790891283056,0.13135506995,0.835878285672\n-2,0.5,mezcla,True,0.719406910792,900,422,0.468888888889,359,312,47,110,431,0.869080779944,0.739336492891,0.839612486545,0.798975672215,0.839612486545,0.120156443786,0.878003194854\n-2,0.5,frecuencia_mensual,False,0.822222222222,900,422,0.468888888889,305,245,60,177,418,0.803278688525,0.580568720379,0.746041412911,0.674002751032,0.746041412911,0.164064263905,0.759418766829\n-2,0.5,persistencia,False,0.5,900,422,0.468888888889,420,323,97,99,381,0.769047619048,0.765402843602,0.768315889629,0.767220902613,0.768315889629,0.217777777778,0.698631234484\n-2,1,logistica,True,0.330740010151,900,422,0.468888888889,527,388,139,34,339,0.73624288425,0.919431279621,0.766798418972,0.8177028451,0.8177028451,0.120035786506,0.879329920995\n-2,1,arbol,False,0.454545454545,900,422,0.468888888889,505,378,127,44,351,0.748514851485,0.895734597156,0.773955773956,0.815533980583,0.815533980583,0.128043427798,0.830093963746\n-2,1,knn,False,0.451612903226,900,422,0.468888888889,468,359,109,63,369,0.767094017094,0.850710900474,0.782476024412,0.806741573034,0.806741573034,0.13135506995,0.835878285672\n-2,1,mezcla,False,0.363549818904,900,422,0.468888888889,517,386,131,36,347,0.746615087041,0.914691943128,0.775100401606,0.822151224707,0.822151224707,0.120156443786,0.878003194854\n-2,1,frecuencia_mensual,False,0.497849462366,900,422,0.468888888889,615,397,218,25,260,0.645528455285,0.940758293839,0.688757807078,0.765670202507,0.765670202507,0.164064263905,0.759418766829\n-2,1,persistencia,False,0.5,900,422,0.468888888889,420,323,97,99,381,0.769047619048,0.765402843602,0.768315889629,0.767220902613,0.767220902613,0.217777777778,0.698631234484\n-4,0.5,logistica,True,0.535830600454,900,255,0.283333333333,227,160,67,95,578,0.704845814978,0.627450980392,0.687876182287,0.663900414938,0.687876182287,0.117457015751,0.718646283643\n-4,0.5,arbol,False,0.532125205931,900,255,0.283333333333,301,206,95,49,550,0.68438538206,0.807843137255,0.705962988348,0.741007194245,0.705962988348,0.12009747462,0.676663541033\n-4,0.5,knn,False,0.483870967742,900,255,0.283333333333,251,172,79,83,566,0.685258964143,0.674509803922,0.683081810961,0.679841897233,0.683081810961,0.125559024165,0.683431614113\n-4,0.5,mezcla,False,0.518533233814,900,255,0.283333333333,242,170,72,85,573,0.702479338843,0.666666666667,0.695012264922,0.684104627767,0.695012264922,0.11617330073,0.723273873955\n-4,0.5,frecuencia_mensual,False,0.666666666667,900,255,0.283333333333,155,89,66,166,579,0.574193548387,0.349019607843,0.508571428571,0.434146341463,0.508571428571,0.161894676756,0.508461161013\n-4,0.5,persistencia,False,0.5,900,255,0.283333333333,255,168,87,87,558,0.658823529412,0.658823529412,0.658823529412,0.658823529412,0.658823529412,0.193333333333,0.530715109573\n-4,1,logistica,True,0.332516889312,900,255,0.283333333333,343,220,123,35,522,0.64139941691,0.862745098039,0.676090964966,0.735785953177,0.735785953177,0.117457015751,0.718646283643\n-4,1,arbol,False,0.532125205931,900,255,0.283333333333,301,206,95,49,550,0.68438538206,0.807843137255,0.705962988348,0.741007194245,0.741007194245,0.12009747462,0.676663541033\n-4,1,knn,False,0.322580645161,900,255,0.283333333333,372,219,153,36,492,0.588709677419,0.858823529412,0.628227194492,0.698564593301,0.698564593301,0.125559024165,0.683431614113\n-4,1,mezcla,False,0.31339732882,900,255,0.283333333333,356,225,131,30,514,0.63202247191,0.882352941176,0.670041691483,0.736497545008,0.736497545008,0.11617330073,0.723273873955\n-4,1,frecuencia_mensual,False,0.312903225806,900,255,0.283333333333,460,209,251,46,394,0.454347826087,0.819607843137,0.498806682578,0.584615384615,0.584615384615,0.161894676756,0.508461161013\n-4,1,persistencia,False,0.5,900,255,0.283333333333,255,168,87,87,558,0.658823529412,0.658823529412,0.658823529412,0.658823529412,0.658823529412,0.193333333333,0.530715109573\n'))
actual=pd.read_csv(OUT/"metricas_prueba.csv")
pd.testing.assert_frame_equal(actual,referencia,check_exact=False,rtol=1e-7,atol=1e-8)
print("VERIFICADO: las metricas de prueba coinciden con las cifras publicadas.")
display(pd.read_csv(OUT/"metricas_prueba.csv").query("thermal == -3 and beta == 0.5"))
print((OUT/"conclusiones_numericas.json").read_text())
shutil.make_archive("resultados_grupo9","zip",OUT)
try:
    from google.colab import files
    files.download("resultados_grupo9.zip")
except ImportError:
    print("Resultados guardados en resultados_grupo9.zip")